# Hidden Markov Models — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def normalize(v):
    v=np.asarray(v,dtype=float); return v/v.sum()
def softmax(v):
    v=np.asarray(v,dtype=float); e=np.exp(v-v.max()); return e/e.sum()
def norm_pdf(x,mu,var):
    return np.exp(-0.5*(np.asarray(x)-mu)**2/var)/np.sqrt(2*np.pi*var)
def beta_pdf_grid(x,a,b):
    B=math.gamma(a)*math.gamma(b)/math.gamma(a+b)
    return (np.asarray(x)**(a-1))*((1-np.asarray(x))**(b-1))/B
print('setup ok')

## Infer hidden state sequences from noisy observations
HMMs combine Markov transitions with emissions. These examples run forward filtering, likelihood computation, Viterbi decoding, smoothing, and the effect of a sticky transition matrix.

In [ ]:
# 1) forward filter after observations 0,1,1
pi=np.array([0.6,0.4]); T=np.array([[0.7,0.3],[0.2,0.8]]); E=np.array([[0.9,0.1],[0.2,0.8]]); obs=[0,1,1]
alpha=pi*E[:,obs[0]]; alpha=alpha/alpha.sum(); hist=[alpha]
for o in obs[1:]: alpha=(alpha@T)*E[:,o]; alpha=alpha/alpha.sum(); hist.append(alpha)
plt.figure(figsize=(6,3)); plt.plot(np.array(hist)); plt.legend(['state0','state1']); plt.title('forward beliefs')
assert alpha[1]>0.8

In [ ]:
# 2) sequence likelihood from unnormalized forward messages
a=pi*E[:,obs[0]]; scales=[a.sum()]; a=a/a.sum()
for o in obs[1:]: a=(a@T)*E[:,o]; scales.append(a.sum()); a=a/a.sum()
lik=np.prod(scales); plt.figure(figsize=(6,3)); plt.bar(['likelihood'],[lik]); plt.title(f'P(obs)={lik:.4f}')
assert abs(lik-0.131542)<1e-9

In [ ]:
# 3) Viterbi path maximizes joint probability
delta=pi*E[:,obs[0]]; ptr=[]
for o in obs[1:]:
    vals=delta[:,None]*T; ptr.append(vals.argmax(axis=0)); delta=vals.max(axis=0)*E[:,o]
path=[int(delta.argmax())]
for p in ptr[::-1]: path.append(int(p[path[-1]]))
path=path[::-1]; plt.figure(figsize=(6,3)); plt.step(range(len(path)),path,where='mid'); plt.ylim(-.2,1.2); plt.title('Viterbi hidden path')
assert path==[0,1,1]

In [ ]:
# 4) backward smoothing changes earlier beliefs using future data
beta=np.ones(2); betas=[beta]
for o in obs[:0:-1]: beta=T@(E[:,o]*beta); betas.append(beta)
b0=hist[0]*betas[-1]; b0=b0/b0.sum(); plt.figure(figsize=(6,3)); plt.bar(['filter S0=1','smooth S0=1'],[hist[0][1],b0[1]]); plt.title('future observations refine the past')
assert b0[1]>hist[0][1]

In [ ]:
# 5) stickier transitions resist switching
stick=np.array([[0.95,0.05],[0.05,0.95]]); a=pi*E[:,0]
for o in [1,1]: a=(a/a.sum()@stick)*E[:,o]
a=a/a.sum(); plt.figure(figsize=(6,3)); plt.bar(['state0','state1'],a); plt.title('sticky dynamics temper evidence')
assert a[1]<alpha[1]